<a href="https://colab.research.google.com/github/Marcin19721205/WSBNeuronowe/blob/main/CW6_Keras_tuner_zadania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wprowadzenie

Poszukiwanie odpowiednich parametrów sieci neuronowej oraz właściwej architektury jest bardzo trudnym zadaniem. Ręczne przeglądanie wszystkich dopuszczalnych konfiguracji paramertrów i połączeń między nimi może być barzdo czasochłonne. Z tego względu, powstały narzędzia takie, jak [Keras Tuner](https://www.tensorflow.org/tutorials/keras/keras_tuner), które pozwalają przeczesywać i wybierać najlepsze architektury sieci dla danego problemu.

W tym notebooku przećwiczymy możliwość poszukiwania odpowiedniego zestawu hiperparametrów z użyciem tego narzędzia.

In [ ]:
%pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.0 MB/s eta 0:00:00


In [ ]:
import gc
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras as krs
import sklearn.datasets as skds
import keras_tuner as kt


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Wczytanie danych

W tym notebooku będziemy pracować z zestawem danych z poprzedniego notebooka, poszukiwanie odpowiedniej architektury pozwoli nam znaleźć taki układ, który zapewni najlepsze wyniki.

1. Dane treningowe znajdują się pod adresem: `https://drive.google.com/uc?id=1THmneUv2PBA-czNr9dMjMJuN88O7kfyJ&export=download`
2. Dane testowe znajdują się pod adresem `https://drive.google.com/uc?id=1MyZOwSy_ccfW8LyxKxfGXldjjNeFGE-w&export=download`

In [ ]:
train_data = pd.read_csv("https://drive.google.com/uc?id=1THmneUv2PBA-czNr9dMjMJuN88O7kfyJ&export=download")
test_data = pd.read_csv("https://drive.google.com/uc?id=1MyZOwSy_ccfW8LyxKxfGXldjjNeFGE-w&export=download")

In [ ]:
X_train, y_train = train_data.drop(columns=['y']), train_data.y
X_test, y_test = test_data.drop(columns=['y']), test_data.y

# Keras tuner - wprowadzenie

Biblioteka Keras Tuner pozwala określić jakie hiperparametry i w jakim zakresie mają być przeszukiwane. Co do zasady potrzebujemy trzech składników:

1. Funkcji budującej sieć neuronową - funkcja przyjmuje obiekt **hp** - czyli zestaw aktualnie przetwarzanych hiperparametrów.
2. Ścieżki, do której zapisywane będą wyniki poszukiwania.
3. Zestawu danych do szkolenia.


Poniższe ćwiczenia pomogą Ci zbudować pętle od prostych (wybieranie tylko ilości neuronów) do bardziej skomplikowanych (wybieranie il. neuronów, il. warstw, optymalizatory, etc.).

TODO: opisać działanie algorytmu Hyperband

# Bardzo prosty tuner

## Funkcja budująca model i wybierająca parametry

W Keras Tunerze musimy wskazać, w jaki sposób mają być wybierane parametry. Dla poszczególnych typów parametrów mamy następujące możliwości:

1. Wartości całkowitoliczbowe (int) - **hp.Int(NAME, min_value=..., max_value=..., step=...)**

    1. **name** - każdy parametr musi mieć swoją unikalną nazwę, po której potem znajdziemy jego optymalną wartość
    2. **min_value** - najmniejsza dopuszczalna wartość parametru
    3. **max_value** - maksymalna wartość przedziału
    4. **step** - inkrementacja w kolejnych iteracjach
<div>
    <br>
</div>
    
2. Wybór z listy dostępnych wartości - **hp.Choice(NAME, values=[...])**
    1. **name** - jak wyżej
    1. **values** - lista dopuszczalnych wartości spośród których można wybierać
    

<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>: uzupełnij implementację funkcji, która będzie bdować prostą sieć neuronową:
    <ol>
        <li>Pierwsza warstwa - <i>Dense</i>: </li>
            <ol>
                <li>liczba neuronów będzie wybierana przez Keras Tuner z przedziału całkowitoliczbowego między 4 a 32, z inkrementacją o 4</li>
                <li>aktywacja będzie wybierana przez Keras Tuner z listy dostępnych opcji: ['tanh', 'relu']</li>        
            </ol>
        <li>Warstawa wyjściowa - <i>Dense</i>:</li>
        <ol>
            <li>Liczba neuronów równa liczbie klas = 3</li>
            <li>Aktywacja - softmax</li>
        </ol>
        <li>Kompilacja:</li>
        <ol>
            <li>Optymalizator - <b>Adam</b></li>
            <li>Funkcja kosztu: <i>sparse_categorical_crossentropy</i></li>
            <li>Metryki: <i>accuracy</i></li>
        </ol>        
    </ol>
</div>

In [ ]:
def build_simple_model(hp):
    model = krs.Sequential()
    model.add(krs.layers.Dense(units=hp.Int('units', min_value=4, max_value=32, step=4),
                                 activation=hp.Choice('activation', values=['tanh', 'relu']),
                                 input_shape=(X_train.shape[1],)))
    model.add(krs.layers.Dense(units=3, activation='softmax'))

    model.compile(optimizer='Adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## Keras tuner - konfiguracja

<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>: Przygotuj obiekt typu <code> keras_tuner.Hyperband </code> przekazując następujące elementy:
    
<ol>
<li>Funkcję do budowy sieci z poprzedniego punktu</li>
<li>Funkcję celu: <code>val_accuracy</code></li>
<li>Max liczba epok = 10 (maksymalna liczba dla każdego pojedynczego modelu)</li>
<li>hyperband_iterations = 2 - liczba iteracji całego algorytmu Hyperband, czyli ile razy powtórzyć całą procedurę. To bardzo kosztowna czasowo operacja, stąd tymzasowo ustalamy na 2</li>
<li>directory - ścieżka do zapisu wyników optymalizacji. <b>UWAGA!!! </b> użytkownicy Windowsa mogą mieć problem z działaniem funkcji jeśli nazwa ścieżki jest za długa. Najlepiej wybierać krótszą niż 100 znaków. Pod Windowsem należy <b>ZNORMALIZOWAĆ ŚCIEŻKĘ DO ISTNIEJĄCEGO KATALOGU</b> wykorzystując os.path.normpath</li>
<li>project_name - dowolna nazwa projektu</li>
</ol>
</div>



<div class='alert alert-block alert-info'>
    Algorytm Hyperband wykorzystuje adaptacyjne przeszukiwanie przestrzeni modeli, starjając się jak najszybciej znaleźć najbardziej obiecujący. Pod tym względem przypomina <b>Algorytm genetyczny</b> utrzymujący w puli tzw. "elitę" czyli najlepsze osobniki.
    <br>
       Hyperband szkoli wiele modeli przez kilka epok, zachowuje do następnej "rundy" tylko połowę najlepszych. W każdej rundzie szkolognych jest <code>1 + log(max epochs)</code> modeli.
    
   <br>
    Więcej szczegółów na temat działania tego algorytmu można odneleźć w oficjalnej publikacji naukowej: <a href="https://arxiv.org/pdf/1603.06560.pdf">link</a>
</div>

In [ ]:
tuner = kt.Hyperband(build_simple_model,
                     objective='val_accuracy',
                     max_epochs=10,
                     hyperband_iterations=2,
                     directory=os.path.normpath('keras_tuner_simple_search'),
                     project_name='simple_nn_tuning')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Keras tuner - prosty wybór parametrów

Mając przygotowane:
1. funkcję budującą model w każdej epoce
2. obiekt algorytmu Hyperband

należy rozpocząć proces poszukiwania architektury.

<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>: Wywołaj funkcję <code>tuner.search</code> przekazując:
    <ol>
        <li> X_train, y_train - dane uczące </li>
        <li> liczba epok - 20, całkowita liczba powtórzeń  </li>
        <li> validation split = 0.2, ilość danych używanych do walidacji</li>
        <li> utwórz i przekaż wywołanie zwrotne (ang. <i>callback</i>) EarlyStopping, obserwujący trafność walidacyjną i pozwalający na 3 rundy bez poprawy</li>
    </ol>
</div>


In [ ]:
early_stopping_callback = krs.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)
tuner.search(X_train, y_train, epochs=20, validation_split=0.2, callbacks=[early_stopping_callback])

Trial 22 Complete [00h 00m 05s]
val_accuracy: 0.6256250143051147

Best val_accuracy So Far: 0.765625
Total elapsed time: 00h 01m 18s


Po zakończeniu procesu uczenia się możemy wyciągnąć zestawy najlepszych parametrów, uszeregowanych wg. uzyskanych wyników.
```
tuner.get_best_hyperparameters(top_N)
```

A następnie sprawdzać je, wyciągając wartości poszczególnych parametrów **wg. nazwy, którą sami nadaliśmy** funkcją ``get('NAZWA')``, np. ``` tuner.get_best_hyperparameters()[0].get('activation') ```. Jako nazwę podajemy parametr, który wcześniej sami nazwaliśmy (w pierwszym ćwiczeniu).

<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>:pobierz jeden najlepszy zestaw parametrów i zbadaj liczbę jednostek oraz aktywację uznane za najlepsze.
</div>

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_activation = best_hps.get('activation')
best_no_units = best_hps.get('units')
print(f"Best activation: {best_activation}, best no. units: {best_no_units}")

Best activation: relu, best no. units: 32


## Budowanie i szkolenie najlepszego modelu

Mając znalezione najlepsze parametry oraz przeszukaną przestrzeń architektur modeli, możemy zbudować i wyszkolić najlepszy model z dostępnych.
W tym celu wykonujemy następujące kroki:

1. Pobieramy najlepsze parametry, jako jeden obiekt, bez wyciągania konkretnych wartości: `` best_params =  tuner.get_best_hyperparameters()[0] ``
1. Z tunera wybieramy ``hypermodel.build(best_params)``, co wymusi skonstruowanie modelu w oparciu o najlepsze paramtery jak dotąd.
2. Tak zbudowany model szkolimy na zwyczajnych zasadach.

<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>:
    <ol>
        <li>zbuduj i wyszkol na danych treningowych model o najlepszych parametrach znalezionych przez tuner - liczbę epok wybierz stosownie do swoich możliwości obliczeniowych</li>
        <li>Możesz wykorzystać <i>callback</i> EarlyStopping z dowolnymi ustawieniami.</li>
        <li>Następnie przeprowadź ewaluację modelu na danych testowych</li>
    </ol>
    
</div>

In [ ]:
best_params = tuner.get_best_hyperparameters(num_trials=1)[0]
model = tuner.hypermodel.build(best_params)

early_stopping_callback = krs.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)
history = model.fit(X_train, y_train, epochs=20, validation_split=0.2, callbacks=[early_stopping_callback])

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3385 - loss: 1.3277 - val_accuracy: 0.4675 - val_loss: 1.0163
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4933 - loss: 0.9965 - val_accuracy: 0.5856 - val_loss: 0.9069
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6029 - loss: 0.8884 - val_accuracy: 0.6400 - val_loss: 0.8292
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6421 - loss: 0.8256 - val_accuracy: 0.6687 - val_loss: 0.7758
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6780 - loss: 0.7637 - val_accuracy: 0.6831 - val_loss: 0.7326
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7041 - loss: 0.7111 - val_accuracy: 0.7106 - val_loss: 0.6926
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7126 - loss: 0.6876 - val_accuracy: 0.7150 - val_loss: 0.6727
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7289 - loss: 0.6572 - val_accuracy: 0.7375 - val_

In [ ]:
model.evaluate(X_test, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7958 - loss: 0.5358


[0.5434945821762085, 0.7935000061988831]

# Tuner wybierający liczbę warstw

Drugim krokiem może być zbudowanie tunera, który będzie wyszukiwał optymalną **liczbę warstw ukrytych** ORAZ **liczbę neuronów + aktywację** dla każdego z nich. Można to uzyskać w stosunkowo prosty sposób: poprzez dwie zagnieżdżone pętle:
1. Pierwsze wyszukiwane szuka liczę warstw ukrytych (numer).
2. Następnie iterujemy od 0 do N-warstw-ukrytych i tworzymy warstwę ukrytą.
    1. Dla takiej warstwy szukamy liczby neurnów.
    2. Szukamy też aktywacji.
    
    
Przykładowo:

```python

    hp_nlayers = hp.Int('nlayers', min_value=1, max_value=3, step=1)                                       # liczba warstw ukrytych - hiperparametr
    for l_idx in range(hp_nlayers):                                                                        # od 0 do n_warstw
        hp_units = hp.Int(f'units_layer_{l_idx}', min_value=4, max_value=32, step=4)                       # liczba neuronów ukrytych - hiperparametr
        hp_activations = hp.Choice(f'activation_layer_{l_idx}', values=['tanh', 'relu'])                   # aktywacja - hiperparametr
        model1.add(krs.layers.Dense(hp_units, hp_activations, input_shape=(X_train.shape[1],)))            # utworzenie i dodanie odpowiedniej warstwy


```

<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>: przeprowadź kompleksowy eksperyment poszukiwania architektury dla dostarczonych danych. Po znalezieniu modelu, przeprowadź jego ewaluację na danych testowych.
</div>

<div class='alert alert-block alert-danger'>
    <b>Uwaga</b> poszukiwanie najlepszej architektury może zająć trochę czasu, więc nie przesadź z ilością iteracji i epok. Wszystko zależy od sprzętu i mocy obliczeniowej, jaką masz do dyspozycji.
</div>

In [ ]:
def complex_architecture_search(hp):
    model = krs.Sequential()

    num_layers = hp.Int('num_layers', min_value=1, max_value=3, step=1)

    for i in range(num_layers):
        units = hp.Int(f'units_layer_{i}', min_value=4, max_value=32, step=4)
        activation = hp.Choice(f'activation_layer_{i}', values=['tanh', 'relu'])

        if i == 0:
            model.add(krs.layers.Dense(units=units, activation=activation, input_shape=(X_train.shape[1],)))
        else:
            model.add(krs.layers.Dense(units=units, activation=activation))

    model.add(krs.layers.Dense(units=3, activation='softmax'))

    model.compile(optimizer='Adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:
tuner2 = kt.Hyperband(complex_architecture_search,
                     objective='val_accuracy',
                     max_epochs=10,
                     hyperband_iterations=2,
                     directory=os.path.normpath('keras_tuner_complex_search'),
                     project_name='complex_nn_tuning')

early_stopping_callback = krs.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

tuner2.search(X_train, y_train, epochs=20, validation_split=0.05, callbacks=[early_stopping_callback])

Reloading Tuner from keras_tuner_complex_search/complex_nn_tuning/tuner0.json


In [ ]:
model.evaluate(X_test, y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7958 - loss: 0.5358


[0.5434945821762085, 0.7935000061988831]